# SDG

In [4]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env.local"))

sys.path.insert(0, str(Path("..").resolve()))

## Personas

In [5]:
from ragas.testset.persona import Persona

personas = [

    Persona(
        name="Direct & Efficient Client",
        role_description=(
            "A clear and goal-oriented customer who asks direct questions about services, prices, "
            "availability, and wants to quickly book, cancel, or reschedule appointments. "
            "Expects concise and accurate responses without unnecessary details."
        ),
    ),

    Persona(
        name="Indecisive & Conversational Client",
        role_description=(
            "A friendly but unsure customer who is not certain which service they need. "
            "They may ask vague questions like 'I want something for my hair' or change their mind mid-conversation. "
            "Requires the assistant to ask clarifying questions and maintain context across multiple turns."
        ),
    ),

    Persona(
        name="Impatient Returning Client",
        role_description=(
            "A customer who already has a booking and wants to modify or cancel it quickly. "
            "May provide incomplete information (e.g., only name or only time) and expects the assistant "
            "to retrieve existing bookings, handle rescheduling correctly, and avoid errors in tool usage."
        ),
    ),

]

## Load Document

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("../data")
loader = TextLoader(DATA_DIR / "data.txt")
documents = loader.load()

## Generate Testset

In [7]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
import openai

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas,
)

testset = generator.generate_with_langchain_docs(documents, testset_size=10)
testset.to_pandas()



Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/5 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Can you provide detailed information about the...,[# Hair Salon – Business Information ## Salon ...,"Mia, the senior stylist at Maison Lumiere, off...",single_hop_specifc_query_synthesizer
1,Wut kin yu tell me abot hair services in Bostu...,[# Hair Salon – Business Information ## Salon ...,"In Boston, at Maison Lumiere, we offer a varie...",single_hop_specifc_query_synthesizer
2,Wut servises do u offer at Maison Lumiere?,[# Hair Salon – Business Information ## Salon ...,Maison Lumiere offers a variety of services in...,single_hop_specifc_query_synthesizer
3,"Can you tell me more about Mia, the stylist at...",[# Hair Salon – Business Information ## Salon ...,"Mia is a senior stylist at Maison Lumiere, spe...",single_hop_specifc_query_synthesizer
4,Can you tell me about Mia's services at the sa...,[# Hair Salon – Business Information ## Salon ...,"Mia is a senior stylist at Maison Lumiere, spe...",single_hop_specifc_query_synthesizer
5,What is the price for a Children’s Haircut at ...,[Service Prices (USD) ### Haircuts - Women’s H...,The price for a Children’s Haircut (for childr...,single_hop_specifc_query_synthesizer
6,What is the price of a Keratin Treatment and w...,[Service Prices (USD) ### Haircuts - Women’s H...,The Keratin Treatment costs $180 and it helps ...,single_hop_specifc_query_synthesizer
7,What is the cost of a Keratin Treatment?,[Service Prices (USD) ### Haircuts - Women’s H...,The cost of a Keratin Treatment is $180.,single_hop_specifc_query_synthesizer
8,"Um, like, what is this Deep Conditioning Treat...",[Service Prices (USD) ### Haircuts - Women’s H...,The Deep Conditioning Treatment is recommended...,single_hop_specifc_query_synthesizer
9,What is the price of a Deep Conditioning Treat...,[Service Prices (USD) ### Haircuts - Women’s H...,The price of a Deep Conditioning Treatment is ...,single_hop_specifc_query_synthesizer


In [8]:
out_path = Path("../data/testset.json")
testset.to_pandas().to_json(out_path, orient="records", indent=2)

# RAGAS baseline

In [9]:
from langchain.agents import create_agent
from lib.agent import retrieve, SYSTEM_PROMPT

ragas_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[retrieve],
    system_prompt=SYSTEM_PROMPT,
)

In [10]:
from langchain_core.messages import AIMessage

test_df = testset.to_pandas()
dataset = []

for index, row in test_df.iterrows():
    question = row["user_input"]
    print(f"[{index+1}/{len(test_df)}] {question[:80]}")
    result = await ragas_agent.ainvoke({"messsages": [{"role": "user", "content": question}]})
    response = ""
    for msg in reversed(result['messages']):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            response = msg.content
            break
    dataset.append({
        "user_input": question,
        "retrieved_contexts": row['reference_contexts'],
        "response": response,
        "reference": row['reference'],
    })

[1/10] Can you provide detailed information about the services offered by Mia at Maison
[2/10] Wut kin yu tell me abot hair services in Bostun, like, wut do yu have for my hai
[3/10] Wut servises do u offer at Maison Lumiere?
[4/10] Can you tell me more about Mia, the stylist at Maison Lumiere, and what services
[5/10] Can you tell me about Mia's services at the salon?
[6/10] What is the price for a Children’s Haircut at your salon, and do you offer any s
[7/10] What is the price of a Keratin Treatment and what does it do for hair?
[8/10] What is the cost of a Keratin Treatment?
[9/10] Um, like, what is this Deep Conditioning Treatment thingy and how much it cost, 
[10/10] What is the price of a Deep Conditioning Treatment?


In [11]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

evaluation_dataset = EvaluationDataset.from_list(dataset)

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), ResponseRelevancy(), ContextPrecision()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_recall': 0.9800, 'faithfulness': 0.0867, 'answer_relevancy': 0.0000, 'context_precision': 1.0000}

In [12]:
baseline_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,context_precision
0,Can you provide detailed information about the...,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","Mia, the senior stylist at Maison Lumiere, off...",1.0,0.000000,0.0,1.0
1,Wut kin yu tell me abot hair services in Bostu...,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","In Boston, at Maison Lumiere, we offer a varie...",0.8,0.000000,0.0,1.0
2,Wut servises do u offer at Maison Lumiere?,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",Maison Lumiere offers a variety of services in...,1.0,0.000000,0.0,1.0
3,"Can you tell me more about Mia, the stylist at...",[# Hair Salon – Business Information ## Salon ...,"Hi — I'm Lumi, Maison Lumiere's booking assist...","Mia is a senior stylist at Maison Lumiere, spe...",1.0,0.000000,0.0,1.0
4,Can you tell me about Mia's services at the sa...,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","Mia is a senior stylist at Maison Lumiere, spe...",1.0,0.000000,0.0,1.0
5,What is the price for a Children’s Haircut at ...,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The price for a Children’s Haircut (for childr...,1.0,0.166667,0.0,1.0
6,What is the price of a Keratin Treatment and w...,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The Keratin Treatment costs $180 and it helps ...,1.0,0.000000,0.0,1.0
7,What is the cost of a Keratin Treatment?,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi — I'm Lumi, Maison Lumiere's booking assist...",The cost of a Keratin Treatment is $180.,1.0,0.000000,0.0,1.0
8,"Um, like, what is this Deep Conditioning Treat...",[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The Deep Conditioning Treatment is recommended...,1.0,0.300000,0.0,1.0
9,What is the price of a Deep Conditioning Treat...,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The price of a Deep Conditioning Treatment is ...,1.0,0.400000,0.0,1.0


# RAGAS Advanced 

In [13]:
from lib.agent import advanced_retrieve, SYSTEM_PROMPT

advanced_ragas_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[advanced_retrieve],
    system_prompt=SYSTEM_PROMPT,
)

In [14]:
from langchain_core.messages import AIMessage

test_df = testset.to_pandas()
dataset = []

for index, row in test_df.iterrows():
    question = row["user_input"]
    print(f"[{index+1}/{len(test_df)}] {question[:80]}")
    result = await advanced_ragas_agent.ainvoke({"messsages": [{"role": "user", "content": question}]})
    response = ""
    for msg in reversed(result['messages']):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            response = msg.content
            break
    dataset.append({
        "user_input": question,
        "retrieved_contexts": row['reference_contexts'],
        "response": response,
        "reference": row['reference'],
    })

[1/10] Can you provide detailed information about the services offered by Mia at Maison
[2/10] Wut kin yu tell me abot hair services in Bostun, like, wut do yu have for my hai
[3/10] Wut servises do u offer at Maison Lumiere?
[4/10] Can you tell me more about Mia, the stylist at Maison Lumiere, and what services
[5/10] Can you tell me about Mia's services at the salon?
[6/10] What is the price for a Children’s Haircut at your salon, and do you offer any s
[7/10] What is the price of a Keratin Treatment and what does it do for hair?
[8/10] What is the cost of a Keratin Treatment?
[9/10] Um, like, what is this Deep Conditioning Treatment thingy and how much it cost, 
[10/10] What is the price of a Deep Conditioning Treatment?


In [15]:
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

evaluation_dataset = EvaluationDataset.from_list(dataset)

custom_run_config = RunConfig(timeout=360)

advanced_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), ResponseRelevancy(), ContextPrecision()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
advanced_result

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_recall': 0.9800, 'faithfulness': 0.0450, 'answer_relevancy': 0.1596, 'context_precision': 1.0000}

In [16]:
advanced_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,context_precision
0,Can you provide detailed information about the...,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","Mia, the senior stylist at Maison Lumiere, off...",1.0,0.00,0.000000,1.0
1,Wut kin yu tell me abot hair services in Bostu...,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","In Boston, at Maison Lumiere, we offer a varie...",0.8,0.00,0.783661,1.0
2,Wut servises do u offer at Maison Lumiere?,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",Maison Lumiere offers a variety of services in...,1.0,0.00,0.000000,1.0
3,"Can you tell me more about Mia, the stylist at...",[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","Mia is a senior stylist at Maison Lumiere, spe...",1.0,0.25,0.812672,1.0
4,Can you tell me about Mia's services at the sa...,[# Hair Salon – Business Information ## Salon ...,"Hi! I'm Lumi, Maison Lumiere's booking assista...","Mia is a senior stylist at Maison Lumiere, spe...",1.0,0.00,0.000000,1.0
5,What is the price for a Children’s Haircut at ...,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I’m Lumi, Maison Lumiere’s booking assista...",The price for a Children’s Haircut (for childr...,1.0,0.20,0.000000,1.0
6,What is the price of a Keratin Treatment and w...,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi — I'm Lumi, Maison Lumiere's booking assist...",The Keratin Treatment costs $180 and it helps ...,1.0,0.00,0.000000,1.0
7,What is the cost of a Keratin Treatment?,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The cost of a Keratin Treatment is $180.,1.0,0.00,0.000000,1.0
8,"Um, like, what is this Deep Conditioning Treat...",[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The Deep Conditioning Treatment is recommended...,1.0,0.00,0.000000,1.0
9,What is the price of a Deep Conditioning Treat...,[Service Prices (USD) ### Haircuts - Women’s H...,"Hi! I'm Lumi, Maison Lumiere's booking assista...",The price of a Deep Conditioning Treatment is ...,1.0,0.00,0.000000,1.0
